# 06. Generative Adversarial Networks (GANs)

Introduced by Ian Goodfellow et al. in 2014, GANs frame generative modeling as a **two-player zero-sum game** between two competing neural networks:

### The Competitors:
1. **Generator ($G$)**: Maps random noise vector $\mathbf{z} \sim \mathcal{N}(0, \mathbf{I})$ to synthetic data instances $G(\mathbf{z})$ attempting to fool the discriminator.
2. **Discriminator ($D$)**: Binary classifier outputting probability $D(\mathbf{x}) \in [0, 1]$ that instance $\mathbf{x}$ is real rather than generated.

### Minimax Objective:
$$\min_G \max_D V(D, G) = \mathbb{E}_{\mathbf{x} \sim p_{\text{data}}}[\log D(\mathbf{x})] + \mathbb{E}_{\mathbf{z} \sim p_{\mathbf{z}}}[\log(1 - D(G(\mathbf{z})))]$$

In [ ]:
import torch
import torch.nn as nn

# Minimal Generator and Discriminator architecture
class Generator(nn.Module):
    def __init__(self, latent_dim=100, out_dim=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, out_dim), nn.Tanh()
        )
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, in_dim=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

G = Generator()
D = Discriminator()
z = torch.randn(4, 100)
fake_images = G(z)
validity = D(fake_images)
print("Generated Fake Image Shape:", fake_images.shape)
print("Discriminator Realness Score:", validity.detach().numpy().round(3).ravel())

## 🎯 Exercises & Practice
1. What is "Mode Collapse" in GAN training, and what architectural enhancement resolved it?

In [ ]:
# Exercise Solution
# Answer: Mode collapse occurs when the Generator discovers a small subset of outputs that consistently 
# fool the Discriminator, producing identical outputs regardless of z. Wasserstein GAN (WGAN) with 
# Earth Mover's Distance and Gradient Penalty resolved this by eliminating the saturating JS-divergence!